In [0]:
# Purpose: Gold layer - governance KPIs, scores, gaps, and business-ready metadata
# Author: Erzen Citaku - Metadata & Catalog Engineer
# Last updated: 2026-06-13
# Dependencies: dbx_metadata_governance_dev.silver.metadata_validated

import dlt
from pyspark.sql import functions as F


def silver():
    return spark.read.table("dbx_metadata_governance_dev.silver.metadata_validated")


  
# Helper: NULL-safe check column accessor.
# Pass = check_<name> is TRUE. Fail/gap = FALSE or NULL.
  
def _check(name):
    return F.coalesce(F.col(f"check_{name}"), F.lit(False))


  
# scores_pre
  
@dlt.table(
    name="scores_pre",
    comment="Gold: pre-calculated governance scoring columns"
)
def scores_pre():

    df = silver()

    return (
        df
        .withColumn(
            "documentation_score",
            F.round(F.col("completeness_pct"), 2)
        )
        .withColumn(
            # dq_invalid_pct is NULL when total_record_count = 0 (silver VAL-07).
            # Coalesce to 0 so quality_score stays 100 rather than NULL.
            "quality_score",
            F.round(100 - F.coalesce(F.col("dq_invalid_pct"), F.lit(0.0)), 2)
        )
        .withColumn(
            # CHK-07: security_classification_present
            "classification_score",
            F.when(_check("security_classification_present"), 100).otherwise(0)
        )
        .withColumn(
            "overall_governance_score",
            F.round(
                (F.col("completeness_pct") * 0.50) +
                ((100 - F.coalesce(F.col("dq_invalid_pct"), F.lit(0.0))) * 0.30) +
                (F.when(_check("security_classification_present"), 100).otherwise(0) * 0.20),
                2
            )
        )
    )


  
# table_governance — unchanged; reads scores_pre which is now clean
  
@dlt.table(
    name="table_governance",
    comment="Gold: governance score by table"
)
def table_governance():

    df = dlt.read("scores_pre")

    return (
        df.groupBy(
            "database_name",
            "schema_name",
            "table_id",
            "table_name",
            "table_desc",
            "data_steward"
        )
        .agg(
            F.count("*").alias("column_count"),
            F.avg("overall_governance_score").alias("table_governance_score"),
            F.avg("documentation_score").alias("avg_documentation_score"),
            F.avg("quality_score").alias("avg_quality_score"),
            F.sum(F.col("pii_flag").cast("int")).alias("pii_column_count"),
            F.sum(F.col("critical_data_element_flag").cast("int")).alias("critical_column_count")
        )
    )


  
# table_structure — unchanged; all columns come from bronze passthrough
  
@dlt.table(
    name="table_structure",
    comment="Gold: database, schema, table, and column structure"
)
def table_structure():

    df = silver()

    return df.select(
        "database_id",
        "database_name",
        "system_name",
        "location",
        "schema_id",
        "schema_name",
        "table_id",
        "table_name",
        "table_desc",
        "column_id",
        "column_name",
        "column_desc",
        "data_type"
    )


  
# profile — unchanged; all columns exist in silver
  
@dlt.table(
    name="profile",
    comment="Gold: metadata profile for columns"
)
def profile():

    df = silver()

    return df.select(
        "database_name",
        "schema_name",
        "table_name",
        "column_id",
        "column_name",
        "data_type",
        "pii_flag",
        "critical_data_element_flag",
        "security_classification",
        "certification_level",
        "achieved_cert_level",
        "term_name",
        "term_description",
        "term_subdomain",
        "data_steward",
        "tag_value",
        "total_record_count",
        "invalid_record_count",
        "dq_invalid_pct",
        "completeness_pct"
    )


  
# dlt_summary
# CHK-09 (invalid_record_ratio_within_threshold) = dq_pass equivalent
# CHK-08 (sensitivity_flags_set) = closest proxy to pii_consistency
  
@dlt.table(
    name="dlt_summary",
    comment="Gold: Delta Live Tables pipeline quality summary"
)
def dlt_summary():

    df = silver()

    return df.agg(
        F.count("*").alias("silver_valid_rows"),
        # CHK-09: invalid record ratio within threshold
        F.sum(F.when( _check("invalid_record_ratio_within_threshold"), 1).otherwise(0)).alias("dq_pass_rows"),
        F.sum(F.when(~_check("invalid_record_ratio_within_threshold"), 1).otherwise(0)).alias("dq_failed_rows"),
        # CHK-08: both pii_flag and critical_data_element_flag are non-null
        F.sum(F.when( _check("sensitivity_flags_set"), 1).otherwise(0)).alias("pii_consistent_rows"),
        F.sum(F.when(~_check("sensitivity_flags_set"), 1).otherwise(0)).alias("pii_inconsistent_rows")
    )


  
# kpi_summary — unchanged; all columns exist in silver
  
@dlt.table(
    name="kpi_summary",
    comment="Gold: executive governance KPI summary"
)
def kpi_summary():

    df = silver()

    return df.agg(
        F.count("*").alias("total_columns"),
        F.countDistinct("table_id").alias("total_tables"),
        F.countDistinct("schema_id").alias("total_schemas"),
        F.countDistinct("database_id").alias("total_databases"),
        F.avg("completeness_pct").alias("avg_completeness_pct"),
        F.avg("dq_invalid_pct").alias("avg_dq_invalid_pct"),
        F.sum(F.col("pii_flag").cast("int")).alias("pii_columns"),
        F.sum(F.col("critical_data_element_flag").cast("int")).alias("critical_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "Certified",  1).otherwise(0)).alias("certified_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "Documented", 1).otherwise(0)).alias("documented_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "Registered", 1).otherwise(0)).alias("registered_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "None",       1).otherwise(0)).alias("unclassified_columns")
    )


  
# governance_gaps
# Gap reasons are now strictly 1-to-1 with the 11 rubric checks.
# Removed: has_term_description, has_table_desc, has_subdomain, has_location,
#          pii_consistency — none of these exist as rubric checks in rubric.yaml.
  
@dlt.table(
    name="governance_gaps",
    comment="Gold: missing governance metadata and failed checks"
)
def governance_gaps():

    df = silver()

    return (
        df
        .withColumn(
            "gap_reason",
            F.concat_ws(
                ", ",
                # CHK-04
                F.when(~_check("column_description_present"),            "Missing column description"),
                # CHK-05
                F.when(~_check("business_term_linked"),                  "Missing business term"),
                # CHK-07
                F.when(~_check("security_classification_present"),       "Missing or invalid security classification"),
                # CHK-06
                F.when(~_check("data_steward_assigned"),                 "Missing data steward"),
                # CHK-11
                F.when(~_check("domain_tag_present"),                    "Missing domain tag"),
                # CHK-08
                F.when(~_check("sensitivity_flags_set"),                 "PII or CDE flag is null"),
                # CHK-10
                F.when(~_check("certification_level_populated"),         "Missing or invalid certification level"),
                # CHK-03
                F.when(~_check("schema_assignment_valid"),               "Invalid schema assignment"),
                # CHK-09
                F.when(~_check("invalid_record_ratio_within_threshold"), "Data quality failed: invalid record ratio exceeds 5%"),
            )
        )
        .filter(F.col("gap_reason") != "")
        .select(
            "database_name",
            "schema_name",
            "table_name",
            "column_id",
            "column_name",
            "data_steward",
            "security_classification",
            "achieved_cert_level",
            "completeness_pct",
            "dq_invalid_pct",
            "gap_reason"
        )
    )